In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
# training file: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [2]:
# read it in to inspect it
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("length of dataset in characters: ", len(text))
print(text[0])

length of dataset in characters:  1115394
F


In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [9]:
# create a simple encode/ decode method
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [
    stoi[c] for c in s
]  # encoder: take a string, output a list of integers
decode = lambda l: "".join(
    [itos[i] for i in l]
)  # decoder: take a list of integers, output a string
print(encode("hii there"))
print(decode(encode("hii there")))


# tokenizer eg: openai's tiktoken

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [10]:
import torch

data = torch.tensor(
    encode(text), dtype=torch.long
)  # create a one dimension tensor (array)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [11]:
# Let's now split up the data into train and validation sets
n = int(0.9 * len(data))  # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [12]:
block_size = 8
train_data[: block_size + 1]  # a chunk of training data set
# each chunk has its context

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [13]:
# spell out the chunk
x = train_data[:block_size]
y = train_data[1 : block_size + 1]
for t in range(block_size):
    context = x[: t + 1]
    target = y[t]
    print(f"when the context is {context} the target is expected to be {target}")

when the context is tensor([18]) the target is expected to be 47
when the context is tensor([18, 47]) the target is expected to be 56
when the context is tensor([18, 47, 56]) the target is expected to be 57
when the context is tensor([18, 47, 56, 57]) the target is expected to be 58
when the context is tensor([18, 47, 56, 57, 58]) the target is expected to be 1
when the context is tensor([18, 47, 56, 57, 58,  1]) the target is expected to be 15
when the context is tensor([18, 47, 56, 57, 58,  1, 15]) the target is expected to be 47
when the context is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is expected to be 58


In [16]:
torch.manual_seed(1337)  # fix the seed to reporoduce the data

batch_size = 4  # how many independent sequences will we process in parallel?
block_size = 8  # what is the maximum context length for predictions?


def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == "train" else val_data
    ix = torch.randint(
        len(data) - block_size, (batch_size,)
    )  # 0 to data length - block-size
    print("ix", ix)  # the seed will make ix's result fixed.
    x = torch.stack([data[i : i + block_size] for i in ix])  # add the arbitray offset
    y = torch.stack(
        [data[i + 1 : i + block_size + 1] for i in ix]
    )  # additional 1 offset
    return x, y


xb, yb = get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

print("----")

# b is row number starts from 0
for b in range(batch_size):  # batch dimension
    # t is column number starts from 0, and the training target can be constructed like
    # in each row first x elements from train data will try to perdict the result at the
    # same row and the same x column
    for t in range(block_size):  # time dimension
        context = xb[b, : t + 1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

ix tensor([ 76049, 234249, 934904, 560986])
inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56,

In [18]:
import torch
import torch.nn as nn
from torch.nn import functional as F


class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx)  # (BxTxC) batch , time , channel

        # pytorch accepts channel as the second param-> (BxT)xC
        # in this case (4,8,65) ==> (32,65)
        B, T, C = logits.shape
        logits = logits.view(B * T, C)
        # (4,8) => 32
        targets = targets.view(B * T)

        # when the result is far away from the target, the cross entropy will grow.
        # reference material https://r23456999.medium.com/%E4%BD%95%E8%AC%82-cross-entropy-%E4%BA%A4%E5%8F%89%E7%86%B5-b6d4cef9189d
        # H(p,q) = -Σ p_i log2(qi)
        loss = F.cross_entropy(logits, targets)

        return logits, loss


torch.manual_seed(1337)


m = BigramLanguageModel(vocab_size)
# __call__ will automatically trigger forward method
logits, loss = m(xb, yb)  # xb, yb  are stacked chunks
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)
